# 🧱 05 · 排序特征工程与样本采样

> 「特征工程决定上限」。本节覆盖排序特征体系、特征交叉、**负样本采样**、**Pairwise 排序**——
> 都是搜广推面试的高频手撕/追问点。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 排序特征有哪些？ | 用户/物品/交叉/上下文特征 |
| 2 | 特征交叉怎么做？ | 笛卡尔积 / 哈希 / FM 隐向量 |
| 3 | 负样本怎么采？ | 随机采样 vs 曝光未点击 |
| 4 | 排序怎么学？ | Pointwise vs Pairwise |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 排序特征体系（必背）

In [2]:
print('''| 类别 | 例子 |
|---|---|
| 用户特征 | 年龄/性别/城市/历史 CTR/近 7 日活跃/品类偏好 |
| 物品特征 | 类目/价格档/发布时间/近 7 日曝光点击/质量分 |
| 交叉特征 | 用户品类偏好 × 物品类目、城市 × 价格档 |
| 上下文特征 | 小时/星期/设备/位置/天气/渠道 |
| 统计特征 | 用户-物品共现、物品近 1h 热度（在线特征） |

注意：在线特征（近 1h 热度）训练/线上口径必须一致，否则特征穿越（label leakage）。
''')

| 类别 | 例子 |
|---|---|
| 用户特征 | 年龄/性别/城市/历史 CTR/近 7 日活跃/品类偏好 |
| 物品特征 | 类目/价格档/发布时间/近 7 日曝光点击/质量分 |
| 交叉特征 | 用户品类偏好 × 物品类目、城市 × 价格档 |
| 上下文特征 | 小时/星期/设备/位置/天气/渠道 |
| 统计特征 | 用户-物品共现、物品近 1h 热度（在线特征） |

注意：在线特征（近 1h 热度）训练/线上口径必须一致，否则特征穿越（label leakage）。



## 2️⃣ 特征交叉手写（构建交叉列）

In [3]:
n = 3000
df = pd.DataFrame({
    'city': rng.integers(0, 5, n),
    'cat': rng.integers(0, 6, n),
    'price': rng.uniform(10, 500, n).round(1),
})
df['city_cat'] = df['city'].astype(str) + '_' + df['cat'].astype(str)
df['price_bin'] = pd.cut(df['price'], bins=5, labels=['P1', 'P2', 'P3', 'P4', 'P5'])
print('交叉后示例:')
print(df.head(5).to_string(index=False))
print('\n城市×类目 组合数:', df['city_cat'].nunique())
print('➡️ 交叉维度会爆炸：工业常用 FM 隐向量或哈希交叉代替全笛卡尔积')

交叉后示例:
 city  cat  price city_cat price_bin
    0    4  416.3      0_4        P5
    3    4  301.1      3_4        P3
    3    5  130.0      3_5        P2
    2    5  375.4      2_5        P4
    2    4   51.4      2_4        P1

城市×类目 组合数: 30
➡️ 交叉维度会爆炸：工业常用 FM 隐向量或哈希交叉代替全笛卡尔积


## 3️⃣ 负样本采样策略对比

In [4]:
n_pos = 1000
# 模拟曝光日志：每个正样本周围有大量未点击曝光
pos = pd.DataFrame({'user': rng.integers(0, 100, n_pos), 'item': rng.integers(0, 500, n_pos), 'y': 1})
neg_all = pd.DataFrame({'user': rng.integers(0, 100, 20000), 'item': rng.integers(0, 500, 20000), 'y': 0})

# 策略 A：随机负采样 1:1
neg_a = neg_all.sample(n=len(pos), random_state=7)
print(f'策略A 随机1:1  负样本数={len(neg_a)}，占比 {len(neg_a)/(len(pos)+len(neg_a)):.0%}')
assert len(neg_a) == n_pos and abs(len(neg_a)/(len(pos)+len(neg_a)) - 0.5) < 0.01

# 策略 B：曝光未点击（全量 20k）
print(f'策略B 曝光全量  负样本数={len(neg_all)}，正负比 1:{len(neg_all)//len(pos)}')

# 策略 C：难负样本 = 曝光未点击中"看起来像正样本"的（演示：取用户历史高相关但没点）
neg_c = neg_all.sample(n=len(pos), random_state=7)
print(f'策略C 难负采样  负样本数={len(neg_c)}，模拟取困难样本')
print('\n➡️ 面试：随机负采样会低估负样本难度；曝光未点击更贴近线上；难负样本提升模型判别力')
assert len(neg_c) == n_pos
print('✅ 三种采样策略对比完成')

策略A 随机1:1  负样本数=1000，占比 50%
策略B 曝光全量  负样本数=20000，正负比 1:20
策略C 难负采样  负样本数=1000，模拟取困难样本

➡️ 面试：随机负采样会低估负样本难度；曝光未点击更贴近线上；难负样本提升模型判别力
✅ 三种采样策略对比完成


## 4️⃣ Pointwise vs Pairwise vs Listwise

In [5]:
# Pointwise：对单样本估计 P(点击)；Pairwise：学"谁该排前"
# 构造 pair：同一 query 下，正样本 vs 负样本
pairs_data = []
for _ in range(2000):
    u = rng.integers(0, 50)
    pos_i = rng.integers(0, 200)
    neg_i = rng.integers(0, 200)
    pairs_data.append((u, pos_i, neg_i))

# 简化演示：pairwise 损失期望"正样本得分 > 负样本得分"
rng4 = np.random.default_rng(3)
s_pos = rng4.normal(0.5, 1, 2000)
s_neg = rng4.normal(-0.5, 1, 2000)
hinge = np.maximum(0, 1 - (s_pos - s_neg))
pos_win = (s_pos > s_neg).mean()
print(f'Pairwise hinge 损失均值: {hinge.mean():.3f}（越低说明排序越正确）')
print(f'正样本分>负样本分比例: {pos_win:.1%}')
assert pos_win > 0.5 and hinge.mean() > 0
print('✅ Pairwise 排序对比完成')
print('\n➡️ 面试：RankNet/LambdaRank、BPR 是经典 pairwise；Listwise 直接优化 NDCG 近似')

Pairwise hinge 损失均值: 0.554（越低说明排序越正确）
正样本分>负样本分比例: 76.8%
✅ Pairwise 排序对比完成

➡️ 面试：RankNet/LambdaRank、BPR 是经典 pairwise；Listwise 直接优化 NDCG 近似


## 5️⃣ 排序评估：线上指标到业务指标

In [6]:
print('''排序质量评估金字塔：
  模型层：AUC / GAUC / NDCG@K（离线）
  行为层：CTR / CVR / 深度（停留、完播）
  业务层：GMV / 留存 / 时长 / NPS
面试点：GAUC 比 AUC 好在哪？(用户级排序更贴近真实)
        NDCG 为什么位置加权？(头部位置价值更高)
        AUC 对阈值不敏感意味着什么？(只看排序不看校准)
''')

排序质量评估金字塔：
  模型层：AUC / GAUC / NDCG@K（离线）
  行为层：CTR / CVR / 深度（停留、完播）
  业务层：GMV / 留存 / 时长 / NPS
面试点：GAUC 比 AUC 好在哪？(用户级排序更贴近真实)
        NDCG 为什么位置加权？(头部位置价值更高)
        AUC 对阈值不敏感意味着什么？(只看排序不看校准)



## 📝 自测清单

- [ ] 能背出排序特征四大类并举例
- [ ] 能手写特征交叉并说明维度爆炸问题
- [ ] 能对比随机负采样/曝光未点击/难负样本
- [ ] 能讲清 Pointwise/Pairwise/Listwise 区别
- [ ] 能解释特征穿越（label leakage）与在线特征口径一致